# Data Pipeline — Amazon Reviews 2023 (Appliances)
Exploratory analysis on a streamed sample, followed by schema design and ingestion.

> **Note:** `datasets` >= 3.x removed support for dataset loading scripts
> (`RuntimeError: Dataset scripts are no longer supported`). The raw `.jsonl` file is
> streamed directly via the `json` builder with an `hf://` path instead.

In [4]:
from datasets import load_dataset

# datasets >= 3.x no longer runs dataset scripts; stream the raw jsonl file directly
reviews_stream = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/Appliances.jsonl",
    split="train",
    streaming=True,
)

sample = []
for i, record in enumerate(reviews_stream):
    if i >= 5000:
        break
    sample.append(record)

print(f"Collected {len(sample)} records")
print(sample[0])

Collected 5000 records
{'rating': 5.0, 'title': 'Work great', 'text': 'work great. use a new one every month', 'images': [], 'asin': 'B01N0TQ0OH', 'parent_asin': 'B01N0TQ0OH', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1519317108692, 'helpful_vote': 0, 'verified_purchase': True}


In [5]:
import pandas as pd
df = pd.DataFrame(sample)

print("Shape:", df.shape)
print("\n--- Null counts ---")
print(df.isnull().sum())
print("\n--- Rating distribution ---")
print(df["rating"].value_counts().sort_index())
print("\n--- Timestamp range ---")
ts = pd.to_datetime(df["timestamp"], unit="ms")
print("min:", ts.min(), "| max:", ts.max())
print("\n--- Text length (chars) ---")
print(df["text"].str.len().describe().round(1))

Shape: (5000, 10)

--- Null counts ---
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

--- Rating distribution ---
rating
1.0     376
2.0     170
3.0     293
4.0     582
5.0    3579
Name: count, dtype: int64

--- Timestamp range ---
min: 2007-03-29 23:05:20 | max: 2023-03-17 07:53:04.630000

--- Text length (chars) ---
count     5000.0
mean       248.8
std        454.4
min          1.0
25%         47.0
50%        116.0
75%        279.0
max      11328.0
Name: text, dtype: float64


## Sample EDA — findings

- **Nulls:** none in the 5k sample; ingest still handles nulls defensively (sample ≠ full set).
- **Rating distribution:** J-shaped (72% five-star, 7.5% one-star) — typical for Amazon reviews.
  Averages alone are misleading; distribution will be reported alongside.
- **Timestamp:** 13-digit epoch **milliseconds** (verified: 2007–2023 range after `unit="ms"`).
  Will be converted to seconds (`// 1000`) at ingestion.
- **Text length:** heavily right-skewed (median 116 chars, max 11,328). Most reviews fit a
  single chunk; a long tail needs chunking. Reviews under ~20 chars will be excluded from
  the retrieval corpus.

In [6]:
import sqlite3
import pandas as pd

df = pd.DataFrame(sample)

# Convert ms -> s. Int64 (nullable) looks like the right choice:
# it can safely hold NaN if any timestamp is missing.
ts_seconds = (pd.to_numeric(df["timestamp"], errors="coerce") // 1000).astype("Int64")

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE reviews_test (ts INTEGER)")
conn.executemany(
    "INSERT INTO reviews_test (ts) VALUES (?)",
    [(v,) for v in ts_seconds],
)

# Diagnosis: what type did SQLite ACTUALLY store?
print(conn.execute(
    "SELECT typeof(ts), COUNT(*) FROM reviews_test GROUP BY typeof(ts)"
).fetchall())

[('blob', 5000)]


In [7]:
import numpy as np

# Link 1: what does the Int64 (nullable) column yield per element?
v = ts_seconds.iloc[0]
print("element type:", type(v))

# Link 2: does Python's sqlite3 recognize it as an int?
print("isinstance int:", isinstance(v, int))

# Link 3: minimal reproduction — numpy.int64 straight into SQLite
c2 = sqlite3.connect(":memory:")
c2.execute("CREATE TABLE t (v INTEGER)")
c2.execute("INSERT INTO t VALUES (?)", (np.int64(5),))
print("np.int64 stored as:", c2.execute("SELECT typeof(v) FROM t").fetchone())

# Contrast: a plain Python int
c2.execute("INSERT INTO t VALUES (?)", (int(5),))
print("all types now:", c2.execute("SELECT typeof(v), COUNT(*) FROM t GROUP BY typeof(v)").fetchall())

element type: <class 'numpy.int64'>
isinstance int: False
np.int64 stored as: ('blob',)
all types now: [('blob', 1), ('integer', 1)]


In [8]:
# Fix: convert each value to a native Python int (None-safe)
ts_fixed = [int(v) if pd.notna(v) else None for v in ts_seconds]

c3 = sqlite3.connect(":memory:")
c3.execute("CREATE TABLE reviews_test (ts INTEGER)")
c3.executemany(
    "INSERT INTO reviews_test (ts) VALUES (?)",
    [(v,) for v in ts_fixed],
)

print(c3.execute(
    "SELECT typeof(ts), COUNT(*) FROM reviews_test GROUP BY typeof(ts)"
).fetchall())

[('integer', 5000)]


## Type-integrity finding: pandas nullable Int64 → SQLite BLOB

**Symptom:** `ts` values stored as BLOB despite the column being declared INTEGER —
silently, with no error. Integer operations (`MIN`, `MAX`, comparisons) break downstream.

**Root cause:** pandas' nullable `Int64` dtype yields `numpy.int64` scalars, which fail
`isinstance(v, int)` — so `sqlite3` doesn't recognize them as integers. Since numpy scalars
support the buffer protocol, sqlite3 falls back to storing their raw 8 bytes as BLOB.
SQLite's type affinity is a suggestion, not a constraint, so the INTEGER declaration
doesn't prevent this.

**Fix:** coerce each value to a native Python `int` before insertion
(`int(v) if pd.notna(v) else None`). Verified: `typeof(ts)` → `integer` for all rows.
The ingestion module will include a regression test for this.